<div align="center">
  <img src="https://raw.githubusercontent.com/NaumanHSA/neurosurfer/main/docs/assets/banner/neurosurfer_banner_white.png" alt="Neurosurfer" width="45%"/>
</div>

<br/>

# 01 — Providers, Agents & RAG

This notebook introduces the three building blocks of Neurosurfer:

1. **LLM Provider** — talk directly to the model (`complete` / `stream`)
2. **Agents** — three types that build on the provider:
   - `Agent` — one-shot call, optional structured output
   - `AgenticLoop` — multi-step autonomous loop with native tool calling
   - `ReactAgent` — text-parsing ReAct loop for models without native tool calling
3. **RAG** — ingest a PDF, retrieve relevant chunks, generate a grounded answer

**Model used:** `google/gemma-4-12b-qat` served via LM Studio on `localhost:1234`

---

## Contents
1. [Setup](#1-setup)
2. [LLM Provider — direct usage](#2-llm-provider--direct-usage)
3. [Agent — one-shot](#3-agent--one-shot)
4. [AgenticLoop — multi-step tool use](#4-agenticloop--multi-step-tool-use)
5. [ReactAgent — text-parsing ReAct](#5-reactagent--text-parsing-react)
6. [RAG — Retrieval-Augmented Generation](#6-rag--retrieval-augmented-generation)

---

## 1. Setup

In [1]:
%load_ext autoreload
%autoreload 2

import sys, os
from pathlib import Path

# Point Python at the repo root when running from tutorials/
repo_root = Path(os.getcwd()).parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import neurosurfer
print(f"neurosurfer {neurosurfer.__version__}")

version,0.2.0 | python 3.11.13
os,Linux 6.17.0-35-generic (x86_64)
torch,2.9.1+cu129 CUDA: yes (12.9)
mps,no (built: False)
transformers,4.57.6 sentEmb 5.1.0
accelerate,1.12.0 bnb 0.47.0
gpu,NVIDIA GeForce RTX 3080 Ti


neurosurfer 0.2.0


In [2]:
# ── LM Studio connection ──────────────────────────────────────────────────────
# Make sure LM Studio is running with the model loaded and the local server ON.
# Server → default port 1234.  Copy the model identifier from the LM Studio UI.

LM_STUDIO_URL   = "http://localhost:1234/v1"
LM_STUDIO_MODEL = "google/gemma-4-12b-qat"   # adjust if your name differs
CONTEXT_WINDOW  = 32_768                       # check your model's context length

# Path to the sample document used in the RAG section
PDF_PATH = repo_root / "data" / "pixelrag-paper.pdf"
print(f"PDF exists: {PDF_PATH.exists()} → {PDF_PATH}")

PDF exists: True → /home/nomi/workspace/neurosurfer/data/pixelrag-paper.pdf


In [3]:
# ── JupyterIO ─────────────────────────────────────────────────────────────────
# Agents need an IOHandler for interactive prompts (plan approval, shell commands,
# out-of-scope writes).  This notebook-friendly version auto-approves everything
# and prints notifications so you can see what's happening.

class JupyterIO:
    """Auto-approving IOHandler for notebook demos."""

    async def ask(self, question: str, options=None) -> str:
        print(f"  [ask] {question}")
        return "yes"

    async def request_plan_approval(self, plan: str) -> tuple[bool, str]:
        print(f"  [plan approved]")
        return True, ""

    async def request_shell_approval(self, command: str, reason: str) -> bool:
        print(f"  [shell approved] {command}")
        return True

    async def request_write_approval(self, path: str, summary: str) -> str:
        print(f"  [write approved] {path}")
        return "once"

    def notify(self, message: str) -> None:
        print(f"  [notice] {message}")

print("JupyterIO ready.")

JupyterIO ready.


---

## 2. LLM Provider — direct usage

The **provider** is the lowest layer — a thin async adapter over the model's API.  
You can use it directly when you don't need agent logic (history management, tool gating, events).

Neurosurfer ships two built-in providers:

| Class | Use with |
|---|---|
| `AnthropicProvider` | Anthropic Claude (cloud) |
| `OpenAICompatProvider` | OpenAI, LM Studio, Ollama, vLLM, llama.cpp — anything on the `/v1` API |

In [4]:
from neurosurfer.llm.providers.openai import OpenAICompatProvider

provider = OpenAICompatProvider(
    model          = LM_STUDIO_MODEL,
    base_url       = LM_STUDIO_URL,
    api_key        = "lm-studio",   # LM Studio ignores this; any non-empty string works
    context_window = CONTEXT_WINDOW,
)

print(f"Provider: {provider.model}")
print(f"Context window: {provider.capabilities.context_window:,} tokens")
print(f"Native tool calling: {provider.capabilities.tool_call_style}")

Provider: google/gemma-4-12b-qat
Context window: 32,768 tokens
Native tool calling: openai


### 2a. Single completion

`provider.complete(messages, system, tools, config)` sends one request and returns a `CanonicalResponse`.

- `messages` — list of `Message` objects (conversation history)
- `system` — the system prompt string (passed out-of-band)
- `tools` — tool schemas (empty list = no tools)
- `config` — `GenerationConfig` (max_tokens, temperature, …)

In [5]:
from neurosurfer.llm.types import Message, GenerationConfig

messages = [
    Message.user_text("What is Retrieval-Augmented Generation? Answer in 2 sentences.")
]
config = GenerationConfig(max_tokens=1024, temperature=0.7)

response = await provider.complete(
    messages = messages,
    system   = "You are a concise AI assistant.",
    tools    = [],
    config   = config,
)

print("Answer:", response.text())
print(f"\nTokens used — input: {response.usage.input_tokens}, output: {response.usage.output_tokens}")

Answer: Retrieval-Augmented Generation (RAG) is a technique that enhances large language models by providing them with relevant information from external knowledge sources. It works by retrieving specific data to ground the model's responses in facts, which helps improve accuracy and reduce hallucinations.

Tokens used — input: 37, output: 377


### 2b. Streaming

`provider.stream()` is an async generator that yields typed events token by token.  
This is how you get live output — same as streaming from the chat UI.

In [7]:
from neurosurfer.llm.types import TextDelta, Done
from IPython.display import display, Markdown, clear_output

messages = [Message.user_text("Explain vector embeddings in 3 bullet points.")]
config   = GenerationConfig(max_tokens=1024, temperature=0.7)

output = ""
md = display(Markdown(""), display_id=True)

async for event in provider.stream(
    messages = messages,
    system   = "You are a concise AI assistant.",
    tools    = [],
    config   = config,
):
    if isinstance(event, TextDelta):
        output += event.text
        md.update(Markdown(output))
    elif isinstance(event, Done):
        print(f"\n\nDone. Tokens: {event.response.usage.input_tokens} in / {event.response.usage.output_tokens} out")

* **Numerical Representation:** They convert complex data (like text, images, or audio) into long lists of numbers called vectors.
* **Spatial Mapping:** These vectors place items in a high-dimensional space where proximity represents similarity (e.g., "king" is mathematically closer to "queen" than to "apple").
* **Semantic Understanding:** They allow machines to capture context and relationships, enabling technologies like semantic search, recommendation engines, and Large Language Models (LLMs).



Done. Tokens: 32 in / 432 out


---

## 3. Agent — one-shot

`Agent` makes **one bounded call** — optionally with tools (one round) or a structured Pydantic output.  
It's the simplest agent: ask once, get an answer back.

### Constructing agents

All three agent types share the same base constructor:

| Parameter | Type | Purpose |
|---|---|---|
| `provider` | `Provider` | The LLM backend |
| `tools` | `ToolPool` | Available tools (use `default_pool()` for all built-ins) |
| `system_prompt` | `str` | The agent's persona / instructions |
| `guardrails` | `Guardrails` | Safety limits (write scope, shell policy, max turns) |
| `io` | `IOHandler` | How to ask the user for approvals |
| `cwd` | `Path` | Working directory for file tools |

In [8]:
from neurosurfer.agents import Agent, Guardrails
from neurosurfer.tools import default_pool

agent = Agent(
    provider      = provider,
    tools         = default_pool(),
    system_prompt = "You are a helpful assistant. Be concise.",
    guardrails    = Guardrails(),   # defaults: write_scope=["**"], max_turns=200
    io            = JupyterIO(),
    verbose       = False,   # this cell renders events manually
    cwd           = repo_root,
)

print("Agent ready:", type(agent).__name__)

Agent ready: Agent


### 3a. Text response

`agent.complete(user_input)` runs to completion and returns the final text directly.

In [9]:
from IPython.display import display, Markdown, clear_output

answer = await agent.complete("What is the capital of France, and what is it famous for?")
display(Markdown(answer))

The capital of France is **Paris**. 

It is world-renowned for its rich history, art, fashion, and gastronomy. Some of its most famous landmarks and characteristics include:

*   **Iconic Landmarks:** The **Eiffel Tower**, the **Louvre Museum** (home to the *Mona Lisa*), the **Notre-Dame Cathedral**, and the **Arc de Triomphe**.
*   **Art and Culture:** It is a global center for high fashion, luxury goods, and fine arts, with numerous world-class museums and galleries.
*   **Gastronomy:** Paris is famous for its diverse culinary scene, ranging from high-end Michelin-starred restaurants to traditional boulangeries (bakeries) and cafes serving pastries like croissants and macarons.
*   **Architecture:** The city is known for its Haussmann-style buildings, wide boulevards, and charming neighborhoods like Montmartre and Le Marais.

### 3b. Structured output

Pass `output_schema=YourModel` at construction time and the agent returns a validated Pydantic model — no JSON parsing, no repair loops needed.  
Neurosurfer uses **native tool-use under the hood** to guarantee valid JSON output.

In [10]:
from pydantic import BaseModel

class MovieReview(BaseModel):
    title: str
    genre: str
    rating: float          # 0.0 – 10.0
    summary: str
    pros: list[str]
    cons: list[str]

structured_agent = Agent(
    provider      = provider,
    tools         = default_pool(),
    system_prompt = "You are a film critic. Return structured reviews.",
    guardrails    = Guardrails(),
    io            = JupyterIO(),
    verbose       = False,   # this cell renders events manually
    cwd           = repo_root,
    output_schema = MovieReview,   # ← structured output
)

review: MovieReview = await structured_agent.complete("Write a review for the movie Inception (2010).")

print(f"Title:   {review.title}")
print(f"Genre:   {review.genre}")
print(f"Rating:  {review.rating}/10")
print(f"Summary: {review.summary}")
print("Pros:")
for p in review.pros:
    print(f"  + {p}")
print("Cons:")
for c in review.cons:
    print(f"  - {c}")

Title:   Inception (2010)
Genre:   Sci-Fi/Action
Rating:  9.2/10
Summary: Christopher Nolan’s 'Inception' is a breathtaking cinematic achievement that seamlessly blends high-concept science fiction with a sophisticated heist narrative. By exploring the architecture of dreams and the malleability of memory, Nolan creates a labyrinthine experience that is as intellectually stimulating as it is visually spectacular. It remains a landmark of modern filmmaking, demanding and rewarding the audience's full attention.
Pros:
  + Stunning visual effects and practical cinematography.
  + Masterful narrative structure and pacing.
  + Hans Zimmer's iconic and atmospheric score.
  + Strong, grounded performances from a talented ensemble cast.
  + Deeply thought-provoking exploration of subconscious reality.
Cons:
  - Dense exposition can be overwhelming for casual viewers.
  - Occasionally sacrifices emotional depth in favor of technical spectacle.
  - The complex rules of the dream world require in

---

## 4. AgenticLoop — multi-step tool use

`AgenticLoop` is the **production multi-step agent**. It runs a loop:

```
while not finished:
    call model → if tool_use: execute tools → append results → repeat
```

It uses the provider's **native function-calling API** — no text parsing, no sentinel leakage.  
The loop ends when the model calls the `finish` tool or reaches `max_turns`.

### Events

The `run()` method is an async generator that emits typed events:

| Event | When |
|---|---|
| `TextDelta` | A chunk of text from the model |
| `ThinkingDelta` | A thinking/reasoning chunk (provider-dependent) |
| `TurnCompleted` | One model call finished (includes token usage) |
| `ToolStarted` | A tool is about to execute |
| `ToolFinished` | A tool returned its result |
| `RunFinished` | The entire run is done (status: completed / max_turns / …) |

In [11]:
from neurosurfer.agents import AgenticLoop, events

loop_agent = AgenticLoop(
    provider      = provider,
    tools         = default_pool(),
    system_prompt = (
        "You are a helpful assistant with access to file and search tools. "
        "Complete the task step by step, then call the finish tool when done."
    ),
    guardrails    = Guardrails(),
    io            = JupyterIO(),
    verbose       = False,   # this cell renders events manually
    cwd           = repo_root,
)

print("AgenticLoop ready.")

AgenticLoop ready.


In [13]:
# Run a simple task: list files in the data/ directory and describe what's there

task = "List the files inside the 'data' directory, take the first file, read it and summarize its contents briefly."

print(f"Task: {task}\n{'─' * 60}")

async for ev in loop_agent.run(task):

    if isinstance(ev, events.TextDelta):
        print(ev.text, end="", flush=True)

    elif isinstance(ev, events.ToolStarted):
        print(f"\n\n  [→ tool] {ev.name}({ev.args})")

    elif isinstance(ev, events.ToolFinished):
        result_preview = ev.result.content[:120].replace("\n", " ")
        suffix = "..." if len(ev.result.content) > 120 else ""
        print(f"  [← result] {result_preview}{suffix}")

    elif isinstance(ev, events.TurnCompleted):
        print(f"\n  [turn] {ev.usage.input_tokens} in / {ev.usage.output_tokens} out tokens")

    elif isinstance(ev, events.RunFinished):
        print(f"\n\n{'─' * 60}")
        print(f"Run finished — status: {ev.status}")
        if ev.report:
            print(f"Report: {ev.report}")

Task: List the files inside the 'data' directory, take the first file, read it and summarize its contents briefly.
────────────────────────────────────────────────────────────

  [turn] 2805 in / 292 out tokens


  [→ tool] list_dir({'path': 'data'})
  [← result] data/pixelrag-paper.pdf

  [turn] 2840 in / 472 out tokens


  [→ tool] python_exec({'code': 'try:\n    import PyPDF2\n    print("PyPDF2 is installed")\nexcept ImportError:\n    print("PyPDF2 is not installed")\n\ntry:\n    import pdfminer\n    print("pdfminer is installed")\nexcept ImportError:\n    print("pdfminer is not installed")\n'})
  [← result] stdout: PyPDF2 is not installed pdfminer is installed  exit_code: 0

  [turn] 2955 in / 360 out tokens


  [→ tool] python_exec({'code': 'from pdfminer.high_level import extract_text\nimport os\n\npdf_path = \'data/pixelrag-paper.pdf\'\nif os.path.exists(pdf_path):\n    text = extract_text(pdf_path)\n    print(text[:2000]) # Print first 2000 characters to summarize\nelse:\n    p

### 4a. Limiting the tool set

You can give the agent only specific tools instead of the full pool.  
Use `pool.select(["tool_name", ...])` to pick a subset.

In [14]:
from neurosurfer.tools import default_pool

# Give the agent only read + finish tools — no writes, no shell
read_only_pool = default_pool().select(["read_file", "list_dir", "search", "finish"])
print("Read-only tool pool:", [t.name for t in read_only_pool.all()])

reader_agent = AgenticLoop(
    provider      = provider,
    tools         = read_only_pool,
    system_prompt = "You are a read-only assistant. You can read files but cannot modify them.",
    guardrails    = Guardrails(),
    io            = JupyterIO(),
    verbose       = False,   # this cell renders events manually
    cwd           = repo_root,
)

task = "Read the README.md file and give me the top 3 key features of Neurosurfer."
print(f"\nTask: {task}\n{'─' * 60}")

async for ev in reader_agent.run(task):
    if isinstance(ev, events.TextDelta):
        print(ev.text, end="", flush=True)
    elif isinstance(ev, events.ToolStarted):
        print(f"\n  [→ {ev.name}]")
    elif isinstance(ev, events.RunFinished):
        print(f"\n\nDone — {ev.status}")

Read-only tool pool: ['read_file', 'list_dir', 'search', 'finish']

Task: Read the README.md file and give me the top 3 key features of Neurosurfer.
────────────────────────────────────────────────────────────

  [→ list_dir]

  [→ read_file]
Based on the `README.md` file, the top 3 key features of Neurosurfer are:

1.  **Native tool-use and Agent diversity**: It supports provider-native function calling for clean structured output, as well as a `ReactAgent` for models that don't have native tool APIs (like Ollama or llama.cpp).
2.  **Workflow Builder**: A feature that allows you to describe a workflow in plain English, which then designs and builds a multi-step Directed Acyclic Graph (DAG) that can run on any provider.
3.  **OpenAI-compatible Gateway**: A drop-in FastAPI server that allows you to register agents as model IDs, providing an OpenAI-compatible interface with features like SSE streaming and request/response hooks.

Done — completed


---

## 5. ReactAgent — text-parsing ReAct

`ReactAgent` implements the **Thought → Action → Observation** loop by *parsing the model's text output* — no native tool-calling API required.

**When to use it:**
- The model doesn't support (or poorly supports) JSON function calling
- You're running a small local model that wasn't trained for tool use
- You want explicit step-by-step reasoning visible in the output

The agent prompts the model to emit:
```
Thought: I need to read the README to understand the project.
Action: read_file
Action Input: {"path": "README.md"}
Observation: <tool result appended here by the agent>
... repeat ...
Thought: I now have enough information to answer.
Final Answer: <the answer>
```

The agent streams the model's reasoning as `ThinkingDelta` events (collapse these to a
"Thinking…" indicator), surfaces each tool call as a `ToolStarted` event with a friendly
`title` (e.g. *"Reading file README.md…"*), and emits only the parsed **Final Answer** as
`TextDelta` — so your UI shows clean progress logs followed by the real answer.

> **Note:** Gemma 4 supports native tool calling, so `AgenticLoop` is the better choice for it.  
> `ReactAgent` is demonstrated here to show the interface — use it when you're running a model that doesn't support function calling.

In [7]:
from neurosurfer.agents import ReactAgent, Guardrails, events
from neurosurfer.tools import default_pool

react_agent = ReactAgent(
    provider      = provider,
    tools         = default_pool().select(["read_file", "list_dir", "finish"]),
    system_prompt = "You are a helpful assistant. Think step by step before acting.",
    guardrails    = Guardrails(max_turns=10),  # keep it short for the demo
    io            = JupyterIO(),
    verbose       = False,   # this cell renders events manually
    cwd           = repo_root,
)

print("ReactAgent ready.")

ReactAgent ready.


In [ ]:
task = "List the files in the current folder, read the important ones, and tell me what this project is about."

print(f"Task: {task}\n{'─' * 60}")

thinking = False  # whether we're currently showing the "Thinking…" indicator

async for ev in react_agent.run(task):

    if isinstance(ev, events.ToolStarted):
        # A tool is about to run — show its friendly, human-readable status line
        # (e.g. "Exploring directory .…", "Reading file README.md…").
        if thinking:
            print()            # close the open "Thinking…" line
            thinking = False
        print(f"🔧 {ev.title or ev.name}")

    elif isinstance(ev, events.ToolFinished):
        preview = ev.result.content[:90].replace("\n", " ")
        suffix = "…" if len(ev.result.content) > 90 else ""
        print(f"   ↳ {preview}{suffix}")

    elif isinstance(ev, events.ThinkingDelta):
        # Intermediate reasoning (Thought/Action scaffolding) — collapse it to a
        # single "Thinking…" indicator instead of dumping the raw text.
        if not thinking:
            print("💭 Thinking…", end="", flush=True)
            thinking = True

    elif isinstance(ev, events.TextDelta):
        # The parsed Final Answer — this is the real, user-facing response. Stream it.
        if thinking:
            print("\n")
            thinking = False
        print(ev.text, end="", flush=True)

    elif isinstance(ev, events.RunFinished):
        print(f"\n\n{'─' * 60}")
        print(f"Run finished — {ev.status}")

---

## 6. RAG — Retrieval-Augmented Generation

RAG lets the model answer questions **grounded in your documents** rather than relying on training data alone.

The pipeline:
```
PDF / file  →  chunk  →  embed  →  vector store
                                        ↓
user query  →  embed  →  similarity search  →  top-k chunks
                                                    ↓
                          LLM(system + chunks + query)  →  answer
```

**What you need:**
```bash
pip install "neurosurfer[rag]"
```
This installs `chromadb`, `sentence-transformers`, and PDF/DOCX/PPTX readers.

### 6a. Initialize the RAG agent

In [6]:
# The RAG section requires the [rag] extra.
# Run this cell once if you haven't installed it yet, then restart the kernel.
# !pip install "neurosurfer[rag]"

import importlib
if importlib.util.find_spec("chromadb") is None:
    raise ImportError(
        "chromadb not found. Install the rag extra first:\n"
        "  pip install 'neurosurfer[rag]'\n"
        "then restart the kernel and re-run from here."
    )
if importlib.util.find_spec("sentence_transformers") is None:
    raise ImportError(
        "sentence-transformers not found. Install the rag extra first:\n"
        "  pip install 'neurosurfer[rag]'\n"
        "then restart the kernel and re-run from here."
    )
print("RAG dependencies are available — proceeding.")

RAG dependencies are available — proceeding.


In [7]:
from neurosurfer.rag.agent import RAGAgent
from neurosurfer.rag.config import RAGAgentConfig
from neurosurfer.embeddings import _LocalEmbedder
from neurosurfer.vectorstores.chroma import ChromaVectorStore

# ── Embedder ─────────────────────────────────────────────────────────────────
# Converts text → dense vectors.  all-MiniLM-L6-v2 is fast and downloads once.
embedder = _LocalEmbedder(model="all-MiniLM-L6-v2")
print("Embedder loaded.")

# ── Vector store ──────────────────────────────────────────────────────────────
# ChromaDB persists embeddings to disk.  clear_collection=True resets on each run.
vectorstore = ChromaVectorStore(
    collection_name  = "pixelrag-paper",
    clear_collection = True,
    persist_directory= str(repo_root / "tutorials" / "rag-storage"),
)
print("Vector store initialized.")

# ── RAG agent ─────────────────────────────────────────────────────────────────
rag = RAGAgent(
    llm         = provider,
    embedder    = embedder,
    vectorstore = vectorstore,
    config      = RAGAgentConfig(
        top_k              = 6,
        temperature        = 0.4,
        max_new_tokens     = 8192,
        clear_collection_on_init = False,  # we already set clear_collection above
    ),
)
print("RAGAgent ready.")

/home/nomi/anaconda3/envs/LLMs/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version 2.9.1+cu129 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info


Embedder loaded.
[Init] ChromaVectorStore initialized with collection: pixelrag-paper
Vector store initialized.
RAGAgent ready.


### 6b. Ingest the document

`rag.ingest(source)` reads the file, splits it into overlapping chunks, embeds each chunk, and stores it in the vector store.  
Only needs to run once per document — the vector store persists to disk.

In [8]:
result = rag.ingest(PDF_PATH)

print(f"Ingestion complete:")
print(f"  Sources processed : {result.get('sources', 'n/a')}")
print(f"  Chunks created    : {result.get('chunks', 'n/a')}")
print(f"  Chunks stored     : {result.get('added', 'n/a')}")

Ingestion complete:
  Sources processed : 1
  Chunks created    : 129
  Chunks stored     : 129


### 6c. Retrieve — search without generating

`rag.retrieve(query)` returns the most relevant chunks **without** calling the LLM.  
Useful for inspecting what context the model will see, or for using the chunks in your own pipeline.

In [9]:
query = "How does PixelRAG handle image indexing and retrieval?"

retrieved = rag.retrieve(user_query=query, top_k=4)

print(f"Query: {query}")
print(f"Context tokens used: {retrieved.context_tokens_used}")
print(f"Chunks retrieved: {len(retrieved.docs)}")
print(f"\n{'─' * 60}")
print("Retrieved context (first 800 chars):")
print(retrieved.context[:800], "...")

[06/24/26 13:58:50] INFO     Retrieved 4 docs — 563 context tokens (budget: 31,577 for generation)

Query: How does PixelRAG handle image indexing and retrieval?
Context tokens used: 563
Chunks retrieved: 4

────────────────────────────────────────────────────────────
Retrieved context (first 800 chars):
Source: 3b7290ab85cdce0d:7eac0bb4
Pixel-based retrieval
PIXELRAG (base)
PIXELRAG

---

Source: c282f85ef88bd674:7eac0bb4
Traﬁlatura
Pixel-based retrieval
PIXELRAG (base)

---

Source: f802a8584eef8ba8:7eac0bb4
that represents websites in their native visual form and performs retrieval and
reading entirely in pixel space, enabling an end-to-end architecture that eliminates
text abstraction. PIXELRAG is, to our knowledge, the ﬁrst pipeline to operate
over a full Wikipedia corpus in this form, scaling to a datastore of 30 million
screenshot images with an efﬁcient visual retrieval index. Built on an existing visual
embedding model (i.e., Qwen3-VL-Embedding), PIXELRAG further ﬁne-tunes this
model on screenshot data with carefully curated contrastive training data. Retrieved
screenshots a

### 6d. Run — full RAG pipeline

`rag.run(user_prompt)` retrieves the relevant chunks and calls the LLM with them injected into the prompt.  
The model's answer is grounded in the document — it cannot hallucinate facts that aren't there.

In [10]:
question = "What indexing method does PixelRAG use and what are its advantages over text-only RAG?"

print(f"Question: {question}\n{'─' * 60}")

rag_response = rag.run(user_prompt=question)

print(rag_response.agent_response.text())
print(f"\n{'─' * 60}")
print(f"Tokens: {rag_response.agent_response.usage.input_tokens} in / {rag_response.agent_response.usage.output_tokens} out")

Question: What indexing method does PixelRAG use and what are its advantages over text-only RAG?
────────────────────────────────────────────────────────────


[06/24/26 13:59:50] INFO     Retrieved 6 docs — 1,986 context tokens (budget: 30,154 for generation)

                    INFO     Generating answer...

PixelRAG uses a visual retrieval index based on the following method:
*   **Visual Embedding:** It tiles screenshots into visual embeddings using a visual embedding model (specifically, the Qwen3-VL-Embedding model, which is further fine-tuned on curated screenshot data).
*   **Approximate Nearest-Neighbor Index:** It builds an approximate nearest-neighbor index to facilitate retrieval.
*   **Runtime Retrieval:** At runtime, the system retrieves the top-K tiles for a query and feeds them directly as pixel inputs to a vision-language reader.

**Advantages over text-only RAG include:**
*   **Superior Performance on Diverse Tasks:** PixelRAG consistently outperforms text-based RAG baselines on both text-centric tasks (like NQ and SimpleQA) and multimodal/agentic benchmarks (such as MMSearch, LiveVQA, and MoNaCo), with accuracy improvements of up to 18.1% over text-based baselines in some cases.
*   **Preservation of Visual Layout:** By operating directly on rendered screenshots, it remove

In [11]:
# Try a few more questions to explore the paper

questions = [
    "What datasets were used to evaluate PixelRAG and what were the key results?",
    "How does PixelRAG compare to baseline methods in terms of retrieval accuracy?",
]

for q in questions:
    print(f"\nQ: {q}")
    print("─" * 60)
    resp = rag.run(user_prompt=q)
    print(resp.agent_response.text())
    print()


Q: What datasets were used to evaluate PixelRAG and what were the key results?
────────────────────────────────────────────────────────────


[06/24/26 14:01:09] INFO     Retrieved 6 docs — 738 context tokens (budget: 31,402 for generation)

                    INFO     Generating answer...

Based on the provided context, the datasets and key results for PixelRAG are as follows:

**Datasets Used:**
*   **Wikipedia:** The approach was scaled to over 30 million tiles covering all of Wikipedia.
*   **SimpleQA:** This dataset was used to provide representative examples for various failure modes.
*   **Text-centric benchmarks:** The system was evaluated on these to compare performance against text-based RAG.

**Key Results:**
*   **Outperformance of Text-based RAG:** The pixel-native pipeline outperforms text-based RAG even on text-centric benchmarks that do not require visual reasoning.
*   **Efficiency and Accuracy:** The advantage of the system grows with VLM progress, where newer models achieve higher accuracy with fewer tokens.
*   **Token Reduction:** Image compression provides up to a 3× token reduction, making PIXELRAG more practical with each new model generation.
*   **Success in Specific Failure Modes:** The context indicates that PIXELRAG succeeds (rank ≤3) in insta

[06/24/26 14:01:33] INFO     Retrieved 6 docs — 1,717 context tokens (budget: 30,423 for generation)

                    INFO     Generating answer...

PixelRAG consistently outperforms both no-retrieval and text-based RAG baselines (specifically Trafilatura and mwparserfromhell) across various benchmarks. Its performance compared to these baselines includes the following:

*   **General Performance:** PIXELRAG improves accuracy by up to 18.1% over text-based baselines.
*   **PIXELRAG (base) vs. Baselines:** Even the base version of PIXELRAG outperforms both text-based baselines on every task, with gains of up to 8.3% in recall and 4.5% in accuracy.
*   **Fine-tuning Gains:** Fine-tuning the model provides an additional 5.3% recall improvement and 5.0% accuracy gain.
*   **Specific Benchmarks:**
    *   **EVQA:** While text retrieval provides little benefit over no retrieval on this benchmark, PIXELRAG improves QA accuracy by up to 18% and increases retrieval recall by over 5×.
    *   **LiveVQA:** PIXELRAG (base) outperforms the text baseline by 11.3% and more than doubles recall without requiring domain-specific training.



---

## Summary

| Class | Use when | Key method |
|---|---|---|
| `OpenAICompatProvider` | You need raw LLM access | `await provider.complete(...)` |
| `Agent` | One call, optional structured output | `await agent.complete(...)` |
| `AgenticLoop` | Multi-step autonomy, native tool calling | `async for ev in agent.run(...)` |
| `ReactAgent` | Text-parsing ReAct, no native function calling | `async for ev in agent.run(...)` |
| `RAGAgent` | Ground answers in your documents | `rag.ingest(...)` then `rag.run(...)` |

---

## What's Next

| # | Notebook | Topic |
|---|---|---|
| **01** | *Providers, Agents & RAG* ← **you are here** | Core building blocks |
| **02** | Custom Tools | Define and register your own tools |
| **03** | Graph Agents | Multi-agent DAG workflows with agents and tools |
| **04** | The Gateway Server | Serve agents as OpenAI-compatible endpoints |